# RAG PDF Question Answering — Google Colab

**PDF → chunks → embeddings → FAISS → Gemini → answer**

This version uses Google Gemini for generation. You will need a Gemini API key.

In [ ]:
!pip -q install pypdf sentence-transformers faiss-cpu google-genai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 50.2 MB/s eta 0:00:00


In [ ]:
import getpass
from google import genai
api_key = getpass.getpass('Paste your Gemini API key: ')
client = genai.Client(api_key=api_key)
print('Gemini client ready.')


Paste your Gemini API key: ··········
Gemini client ready.


In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print('Uploaded:', pdf_path)


Saving Administer a Cluster _ Kubernetes.pdf to Administer a Cluster _ Kubernetes.pdf
Uploaded: Administer a Cluster _ Kubernetes.pdf


In [ ]:
from pypdf import PdfReader
reader = PdfReader(pdf_path)
pages = []
for page_number, page in enumerate(reader.pages, 1):
    text = page.extract_text() or ''
    if text.strip():
        pages.append({'page': page_number, 'text': text})
print('Pages with text:', len(pages))
print('Characters:', sum(len(p['text']) for p in pages))
if not pages:
    raise ValueError('No text was extracted. This may be a scanned PDF and needs OCR.')
print(pages[0]['text'][:1500])


Pages with text: 378
Characters: 519439
Administer a Cluster
Learn common tasks for administering a cluster.
1: Administration with kubeadm
1.1: Adding Linux worker nodes
1.2: Adding Windows worker nodes
1.3: Upgrading kubeadm clusters
1.4: Upgrading Linux nodes
1.5: Upgrading Windows nodes
1.6: Certificate Management with kubeadm
1.7: Reconfiguring a kubeadm cluster
1.8: Changing The Kubernetes Package Repository
2: Overprovision Node Capacity For A Cluster
3: Migrating from dockershim
3.1: Changing the Container Runtime on a Node from Docker Engine to
containerd
3.2: Find Out What Container Runtime is Used on a Node
3.3: Troubleshooting CNI plugin-related errors
3.4: Check whether dockershim removal affects you
3.5: Migrating telemetry and security agents from dockershim
4: Generate Certificates Manually
5: Manage Memory, CPU, and API Resources
5.1: Configure Default Memory Requests and Limits for a Namespace
5.2: Configure Default CPU Requests and Limits for a Namespace
5.3: Configu

In [ ]:
def make_chunks(pages, chunk_size=700, overlap=100):
    chunks=[]
    for page in pages:
        words=page['text'].split()
        start=0
        while start < len(words):
            end=min(start+chunk_size, len(words))
            text=' '.join(words[start:end]).strip()
            if text:
                chunks.append({'text':text,'page':page['page']})
            if end == len(words): break
            start=end-overlap
    return chunks
chunks=make_chunks(pages)
print('Chunks:', len(chunks))


Chunks: 378


In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
texts=[c['text'] for c in chunks]
embeddings=embedding_model.encode(texts,convert_to_numpy=True,normalize_embeddings=True,show_progress_bar=True).astype('float32')
print('Embedding shape:', embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Embedding shape: (378, 384)


In [ ]:
import faiss
index=faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print('FAISS vectors:', index.ntotal)


FAISS vectors: 378


In [ ]:
def retrieve(question, top_k=5):
    q=embedding_model.encode([question],convert_to_numpy=True,normalize_embeddings=True).astype('float32')
    scores, ids=index.search(q, min(top_k,len(chunks)))
    return [{'score':float(score),'text':chunks[int(idx)]['text'],'page':chunks[int(idx)]['page']} for score,idx in zip(scores[0],ids[0])]


In [ ]:
def ask_rag(question, top_k=5):
    results=retrieve(question,top_k)
    context='\n\n'.join(f"[Page {r['page']}]\n{r['text']}" for r in results)
    prompt=f'''Answer the question using ONLY the context below. If the answer is not in the context, say you could not find it in the document. Do not invent facts.\n\nCONTEXT:\n{context}\n\nQUESTION:\n{question}\n\nANSWER:'''
    response=client.models.generate_content(model='gemini-3.8-flash',contents=prompt)
    return response.text, results

answer,sources=ask_rag('What is the main topic of this document?')
print(answer)


Based on the document headers and content, the main topic is **Administer a Cluster** (in Kubernetes).


In [ ]:
question=input('Ask a question about your PDF: ')
answer,sources=ask_rag(question)
print('\nANSWER:\n',answer)
print('\nSOURCES:')
for i,s in enumerate(sources,1): print(f"{i}. Page {s['page']} | similarity={s['score']:.3f}")


Ask a question about your PDF: who is US president ?

ANSWER:
 I could not find it in the document.

SOURCES:
1. Page 14 | similarity=0.055
2. Page 65 | similarity=0.045
3. Page 305 | similarity=0.036
4. Page 66 | similarity=0.036
5. Page 182 | similarity=0.034


In [ ]:
while True:
    question=input("\nQuestion (type 'exit' to stop): ").strip()
    if question.lower()=='exit': break
    if not question: continue
    answer,sources=ask_rag(question)
    print('\nAnswer:\n',answer)
    print('Sources:', ', '.join(f"page {s['page']}" for s in sources))



Question (type 'exit' to stop): workload

Answer:
 Based on the provided context, workloads are mentioned in scenarios involving resource constraints across separate production and development namespaces:

* **CPU:** Production workloads can consume up to 3 CPU, while development workloads are limited to 1 CPU (Page 119).
* **Memory:** Production workloads can consume up to 8 GiB of memory, while development workloads are limited to 512 MiB (Page 110).
Sources: page 119, page 156, page 110, page 101, page 130

Question (type 'exit' to stop): nodes in cluster ?

Answer:
 I could not find it in the document.
Sources: page 167, page 164, page 329, page 273, page 331

Question (type 'exit' to stop): nodes 

Answer:
 Based on the provided context, "nodes" appears in the following contexts:

1. **DNS Horizontal Autoscaling (Page 164):** 
   - `nodes` is a parameter in the equation used to calculate the number of DNS backend replicas:  
     `replicas = max( ceil( cores × 1/coresPerReplica )